# State Labels by Industry

data source: https://apps.bea.gov/itable/?ReqID=70&step=1&_gl=1*19kveko*_ga*MTI3MzUwNTg4My4xNzczNTEzNzg2*_ga_J4698JNNFT*czE3NzM1MTM3ODYkbzEkZzEkdDE3NzM1MTcwNjgkajYwJGwwJGgw

Quarterly Gross Domestic Product (GDP) by State

SQGDP2 GDP by industry in current dollars

all areas, all statistics in the table, levels, 2024

In [15]:
# import data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(r'E:\Erdos_DS_BootCamp\Project_my_work\State_labeling_data\SQGDP2_2024.csv', skiprows=3)

In [16]:
df.sample(5)

,GeoFIPS,GeoName,LineCode,Description,2024:Q1,2024:Q2,2024:Q3,2024:Q4
606,25000,Massachusetts,34.0,Wholesale trade,36161.0,36562.9,36693.1,36168.8
708,29000,Missouri,6.0,"Mining, quarrying, and oil and gas extra...",1315.1,1147.7,1072.5,1021.1
1100,44000,Rhode Island,65.0,Administrative and support and waste man...,2688.0,2706.1,2774.8,2754.4
1143,46000,South Dakota,12.0,Manufacturing,5402.3,5525.5,5540.8,5505.6
1332,54000,West Virginia,12.0,Manufacturing,8722.0,8996.2,8903.4,9078.5


In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1628 entries, 0 to 1627
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   GeoFIPS      1628 non-null   str    
 1   GeoName      1623 non-null   str    
 2   LineCode     1622 non-null   float64
 3   Description  1623 non-null   str    
 4   2024:Q1      1622 non-null   str    
 5   2024:Q2      1622 non-null   str    
 6   2024:Q3      1622 non-null   str    
 7   2024:Q4      1622 non-null   str    
dtypes: float64(1), str(7)
memory usage: 221.5 KB


In [18]:
# Preprocessing

# remove regions that are not states
drop_codes_region = ['00000', '91000', '92000', '93000', '94000', '95000', '96000', '97000', '98000']
df = df[~df["GeoFIPS"].isin(drop_codes_region)].copy()

# convert quarters to numeric
quarter_cols = ["2024:Q1", "2024:Q2", "2024:Q3", "2024:Q4"]
df[quarter_cols] = df[quarter_cols].apply(pd.to_numeric, errors="coerce")

# compute annual GDP
df["GDP_2024"] = df[quarter_cols].sum(axis=1)

# subtract government (LineCode 83) from total GDP (LineCode 1) on the original df
gov = df[df["LineCode"] == 83][["GeoFIPS", "GDP_2024"] + quarter_cols]

tot = df[df["LineCode"] == 1].merge(gov, on="GeoFIPS", how="left", suffixes=("", "_gov"))

for c in quarter_cols + ["GDP_2024"]:
    df.loc[df["LineCode"] == 1, c] = (
        tot[c] - tot[f"{c}_gov"].fillna(0)
    ).values

# now remove supersets / subsets / government / addenda rows
drop_codes_industry = [2, 13, 25, 83, 84, 85, 86, 100, 101]
df = df[~df["LineCode"].isin(drop_codes_industry)].copy()

# separate adjusted total GDP
total_gdp = (
    df[df["LineCode"] == 1][["GeoName", "GDP_2024"]]
    .rename(columns={"GDP_2024": "TotalGDP"})
)

industries = df[df["LineCode"] != 1].copy()
industries = industries.merge(total_gdp, on="GeoName", how="left")

df.info()

<class 'pandas.DataFrame'>
Index: 1025 entries, 30 to 1627
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   GeoFIPS      1025 non-null   str    
 1   GeoName      1020 non-null   str    
 2   LineCode     1020 non-null   float64
 3   Description  1020 non-null   str    
 4   2024:Q1      1002 non-null   float64
 5   2024:Q2      1002 non-null   float64
 6   2024:Q3      1002 non-null   float64
 7   2024:Q4      1002 non-null   float64
 8   GDP_2024     1025 non-null   float64
dtypes: float64(6), str(3)
memory usage: 130.5 KB


In [19]:
# Label each state 
industries["share"] = industries["GDP_2024"] / industries["TotalGDP"]
labels = industries.loc[
    industries.groupby("GeoName")["share"].idxmax(),
    ["GeoName","Description","share"]
]

In [20]:
labels

,GeoName,Description,share
4,Alabama,Manufacturing,0.180953
26,Alaska,Transportation and warehousing,0.176181
48,Arizona,Real estate and rental and leasing,0.193636
61,Arkansas,Manufacturing,0.151158
86,California,Real estate and rental and leasing,0.159419
105,Colorado,Real estate and rental and leasing,0.178901
124,Connecticut,Real estate and rental and leasing,0.146056
142,Delaware,Finance and insurance,0.275796
163,District of Columbia,"Professional, scientific, and technical ...",0.302071
181,Florida,Real estate and rental and leasing,0.225947


In [21]:
label_groups = labels.groupby("Description")["GeoName"].apply(list)

for industry, states in label_groups.items():
    print(industry)
    print(states)
    print()

      Finance and insurance
['Delaware', 'Nebraska', 'New York', 'South Dakota']

      Information
['Washington']

      Manufacturing
['Alabama', 'Arkansas', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Michigan', 'Mississippi', 'Ohio', 'Wisconsin']

      Mining, quarrying, and oil and gas extraction
['New Mexico', 'North Dakota', 'West Virginia', 'Wyoming']

      Professional, scientific, and technical services
['District of Columbia', 'Massachusetts']

      Real estate and rental and leasing
['Arizona', 'California', 'Colorado', 'Connecticut', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Maine', 'Maryland', 'Minnesota', 'Missouri', 'Montana', 'Nevada', 'New Hampshire', 'New Jersey', 'North Carolina', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia']

      Transportation and warehousing
['Alaska']



In [22]:
labels["Description"].value_counts()

Description
Real estate and rental and leasing                  28
Manufacturing                                       11
Finance and insurance                                4
Mining, quarrying, and oil and gas extraction        4
Professional, scientific, and technical services     2
Transportation and warehousing                       1
Information                                          1
Name: count, dtype: int64

## Result analysis:

Real estate and rental and leasing is the leading industry for 28 states, so I'll remove it.

In [23]:
# substract real estate GDP from total GDP, do everything in a new df

df_new = df.copy()

# get real estate rows
re = df_new[df_new["LineCode"] == 56][["GeoFIPS", "GDP_2024"] + quarter_cols]

# subtract it from total GDP (LineCode 1)
tot = df_new[df_new["LineCode"] == 1].merge(re, on="GeoFIPS", how="left", suffixes=("", "_re"))

for c in quarter_cols + ['GDP_2024']:
    df_new.loc[df_new["LineCode"] == 1, c] = (
        tot[c] - tot[f"{c}_re"].fillna(0)
    ).values

# remove real estate rows
df_new = df_new[df_new["LineCode"] != 56]


In [24]:
df.head(5)

,GeoFIPS,GeoName,LineCode,Description,2024:Q1,2024:Q2,2024:Q3,2024:Q4,GDP_2024
30,01000,Alabama,1.0,All industry total,269601.8,274613.1,278930.4,282040.3,1105185.6
32,01000,Alabama,3.0,"Agriculture, forestry, fishing and hunting",3334.2,3025.4,4304.2,4779.4,15443.2
33,01000,Alabama,6.0,"Mining, quarrying, and oil and gas extra...",2570.9,2695.7,2551.1,2522.6,10340.3
34,01000,Alabama,10.0,Utilities,8282.3,8485.1,8377.6,8550.9,33695.9
35,01000,Alabama,11.0,Construction,15405.3,15877.5,16090.5,16229.7,63603.0


In [25]:
df_new.head(5)

,GeoFIPS,GeoName,LineCode,Description,2024:Q1,2024:Q2,2024:Q3,2024:Q4,GDP_2024
30,01000,Alabama,1.0,All industry total,230225.4,234588.0,238449.8,241233.6,944496.8
32,01000,Alabama,3.0,"Agriculture, forestry, fishing and hunting",3334.2,3025.4,4304.2,4779.4,15443.2
33,01000,Alabama,6.0,"Mining, quarrying, and oil and gas extra...",2570.9,2695.7,2551.1,2522.6,10340.3
34,01000,Alabama,10.0,Utilities,8282.3,8485.1,8377.6,8550.9,33695.9
35,01000,Alabama,11.0,Construction,15405.3,15877.5,16090.5,16229.7,63603.0


In [26]:
# do labeling on new df

total_gdp_new = df_new[df_new["LineCode"] == 1][["GeoName","GDP_2024"]]
total_gdp_new = total_gdp_new.rename(columns={"GDP_2024":"TotalGDP"})
industries_new = df_new[df_new["LineCode"] != 1].copy()
industries_new = industries_new.merge(total_gdp_new, on="GeoName")

industries_new["share"] = industries_new["GDP_2024"] / industries_new["TotalGDP"]
labels_new = industries_new.loc[
    industries_new.groupby("GeoName")["share"].idxmax(),
    ["GeoName","Description","share"]
]


In [27]:
label_groups_new = labels_new.groupby("Description")["GeoName"].apply(list)

for industry, states in label_groups_new.items():
    print(industry)
    print(states)
    print()

      Accommodation and food services
['Hawaii', 'Nevada']

      Finance and insurance
['Connecticut', 'Delaware', 'Nebraska', 'New York', 'South Dakota']

      Health care and social assistance
['Arizona', 'Maine', 'Montana', 'Rhode Island', 'Vermont']

      Information
['California', 'Washington']

      Manufacturing
['Alabama', 'Arkansas', 'Georgia', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'North Carolina', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'South Carolina', 'Tennessee', 'Texas', 'Utah', 'Wisconsin']

      Mining, quarrying, and oil and gas extraction
['New Mexico', 'North Dakota', 'West Virginia', 'Wyoming']

      Professional, scientific, and technical services
['Colorado', 'District of Columbia', 'Florida', 'Maryland', 'Massachusetts', 'New Hampshire', 'New Jersey', 'Virginia']

      Transportation and warehousing
['Alaska']



In [28]:
labels_new["Description"].value_counts()

Description
Manufacturing                                       24
Professional, scientific, and technical services     8
Health care and social assistance                    5
Finance and insurance                                5
Mining, quarrying, and oil and gas extraction        4
Information                                          2
Accommodation and food services                      2
Transportation and warehousing                       1
Name: count, dtype: int64

### Final Result: Dominant Industry by State (2024)

- **Accommodation & Food Services (2):** Hawaii, Nevada  
- **Finance & Insurance (5):** Connecticut, Delaware, Nebraska, New York, South Dakota  
- **Health Care & Social Assistance (5):** Arizona, Maine, Montana, Rhode Island, Vermont  
- **Information (2):** California, Washington  
- **Manufacturing (24):** Alabama, Arkansas, Georgia, Idaho, Illinois, Indiana, Iowa, Kansas, Kentucky, Louisiana, Michigan, Minnesota, Mississippi, Missouri, North Carolina, Ohio, Oklahoma, Oregon, Pennsylvania, South Carolina, Tennessee, Texas, Utah, Wisconsin  
- **Mining, Quarrying & Oil/Gas (4):** New Mexico, North Dakota, West Virginia, Wyoming  
- **Professional, Scientific & Technical Services (8):** Colorado, District of Columbia, Florida, Maryland, Massachusetts, New Hampshire, New Jersey, Virginia  
- **Transportation & Warehousing (1):** Alaska
